# **Phase 3A:  The 2018 Audit**

## **Step 0: Import all Necessary Libraries**

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
from pathlib import Path
from IPython.display import (Markdown, 
                             display)

sys.path.append("..")

from src.clean import (
    load_clean,
    CLEAN_FILE,
    CLOSED_FILE,
    CLOSED_LABEL
    )

from src.analysis import (HOLIDAYS,
                          daily_table,
                          to_full_calendar,
                          headline_figures,
                          rolling_average)

CLEAN_DIR = Path("../data/clean")
SOURCE_FILE = str("SeoulBikeData.csv")
TABLE_DIR = Path("../reports/tables")

## **Step 1: Load and Filter the Data**

In [2]:
clean_df = load_clean(CLEAN_DIR / CLEAN_FILE)
closed_df = load_clean(CLEAN_DIR / CLOSED_FILE)

# mask out the year
is_2018 = clean_df["year"] == 2018
is_closed18 = closed_df["year"] == 2018

# filter out the masked data
df18 = clean_df[is_2018].copy()
closed18 = closed_df[is_closed18].copy()

In [8]:
closed18["date"].nunique()

13

## **Step 2: Assertions**

In [3]:
# Expected values

EXPECTED_START_DATE = pd.Timestamp("2018-01-01")
EXPECTED_END_DATE = pd.Timestamp("2018-11-30")

EXPECTED_ROWS = 7721                        # 322 * 24 - 7, where 7 is hours out in oct 6
EXPECTED_UNIQUE_DATES = 322

EXPECTED_CLOSED_ROWS = 295
EXPECTED_CLOSED_DATES = 13
EXPECTED_CLOSED_ROWS_IN_DF18 = 0

EXPECTED_ABSENT_DATES = {
    "2018-04-11",
    "2018-05-10",
    "2018-09-18",
    "2018-09-19",
    "2018-09-28",
    "2018-09-30",
    "2018-10-02",
    "2018-10-04",
    "2018-10-09",
    "2018-11-03",
    "2018-11-06",
    "2018-11-09",
}

EXPECTED_PARTIAL_DATES = 1 
EXPECTED_PARTIAL_DATE = pd.Timestamp("2018-10-06")
EXPECTED_PARTIAL_DATE_ROWS = 17 
EXPECTED_CLOSED_RENTALS = 0 
EXPECTED_FULL_HOURS = 8016          # 334 * 24 
EXPECTED_TOTAL_DATES = 334


EXPECTED_MIN_HOUR = 0
EXPECTED_MAX_HOUR = 23
EXPECTED_DISTINCT_HOURS = 24

EXPECTED_SEASON_DATES = {
    "Winter": 59,
    "Spring": 90,
    "Summer": 92,
    "Autumn": 81,
}

EXPECTED_MONTH_DATES = {
    1: 31,
    2: 28,
    3: 31,
    4: 29,
    5: 30,
    6: 30,
    7: 31,
    8: 31,
    9: 26,
    10: 28,
    11: 27,
}

EXPECTED_MIN_WEEK = 5
EXPECTED_MAX_WEEK = 53
EXPECTED_WEEK_5_DATES = 4
EXPECTED_WEEK_53_DATES = 1

EXPECTED_MISSING_VALUES = 0
EXPECTED_MIN_RENTALS = 0
EXPECTED_DUPLICATE_ROW = 0
EXPECTED_DUPLICATE_DATE_HOUR = 0


# Date range assertions

assert df18["date"].min() == EXPECTED_START_DATE, (
    f"Start date mismatch: EXPECTED {EXPECTED_START_DATE}, "
    f"Got {df18['date'].min()}"
)

assert df18["date"].max() == EXPECTED_END_DATE, (
    f"End date mismatch: EXPECTED {EXPECTED_END_DATE}, "
    f"Got {df18['date'].max()}"
)


# Row and unique date assertions

assert len(df18) == EXPECTED_ROWS, (
    f"Row count mismatch: EXPECTED {EXPECTED_ROWS}, Got {len(df18)}"
)

assert df18["date"].nunique() == EXPECTED_UNIQUE_DATES, (
    f"Unique date count mismatch: EXPECTED {EXPECTED_UNIQUE_DATES}, "
    f"Got {df18['date'].nunique()}"
)


# Closed records assertions

assert len(closed18) == EXPECTED_CLOSED_ROWS, (
    f"Closed row count mismatch: EXPECTED {EXPECTED_CLOSED_ROWS}, "
    f"Got {len(closed18)}"
)

assert closed18["date"].nunique() == EXPECTED_CLOSED_DATES, (
    f"Closed date count mismatch: EXPECTED {EXPECTED_CLOSED_DATES}, "
    f"Got {closed18['date'].nunique()}"
)

assert (df18["functioning_day"] == CLOSED_LABEL).sum() == EXPECTED_CLOSED_ROWS_IN_DF18, (
    f"Closed hours inside df18 mismatch: EXPECTED {EXPECTED_CLOSED_ROWS_IN_DF18}, "
    f"Got {(df18['functioning_day'] == CLOSED_LABEL).sum()}"
)

# Closed rows contain zero rentals 
closed_rentals = closed18["rented_bike_count"].sum()

assert closed_rentals == EXPECTED_CLOSED_RENTALS, ( 
    f"Closed rentals mismatch: EXPECTED {EXPECTED_CLOSED_RENTALS}, " 
    f"Got {closed_rentals}" )


# Fully closed dates must be absent from df18

actual_dates = set(df18["date"].dt.strftime("%Y-%m-%d"))

assert EXPECTED_ABSENT_DATES.isdisjoint(actual_dates), (
    "Some fully closed dates are present in df18: "
    f"{EXPECTED_ABSENT_DATES.intersection(actual_dates)}"
)


# Hour assertions

assert df18["hour"].min() == EXPECTED_MIN_HOUR, (
    f"Minimum hour mismatch: EXPECTED {EXPECTED_MIN_HOUR}, "
    f"Got {df18['hour'].min()}"
)

assert df18["hour"].max() == EXPECTED_MAX_HOUR, (
    f"Maximum hour mismatch: EXPECTED {EXPECTED_MAX_HOUR}, "
    f"Got {df18['hour'].max()}"
)

assert df18["hour"].nunique() == EXPECTED_DISTINCT_HOURS, (
    f"Distinct hour count mismatch: EXPECTED {EXPECTED_DISTINCT_HOURS}, "
    f"Got {df18['hour'].nunique()}"
)


# Season assertions

actual_season_dates = (
    df18.groupby("seasons")["date"]
    .nunique()
    .to_dict()
)

assert actual_season_dates == EXPECTED_SEASON_DATES, (
    f"Season date counts mismatch: EXPECTED {EXPECTED_SEASON_DATES}, "
    f"Got {actual_season_dates}"
)


# Month assertions

actual_month_dates = (
    df18.groupby("month")["date"]
    .nunique()
    .to_dict()
)

assert actual_month_dates == EXPECTED_MONTH_DATES, (
    f"Month date counts mismatch: EXPECTED {EXPECTED_MONTH_DATES}, "
    f"Got {actual_month_dates}"
)

# Exactly one partial date exists 

rows_per_date = df18.groupby("date")["hour"].count() 
partial_dates = rows_per_date[rows_per_date < EXPECTED_DISTINCT_HOURS]

assert len(partial_dates) == EXPECTED_PARTIAL_DATES, (
    f"Partial date count mismatch: EXPECTED {EXPECTED_PARTIAL_DATES}, "
    f"Got {len(partial_dates)}"
) 
assert partial_dates.index[0] == EXPECTED_PARTIAL_DATE, (
    f"Partial date mismatch: EXPECTED {EXPECTED_PARTIAL_DATE}, "
    f"Got {partial_dates.index[0]}"
)
assert partial_dates.iloc[0] == EXPECTED_PARTIAL_DATE_ROWS, (
    f"Partial date row count mismatch: EXPECTED {EXPECTED_PARTIAL_DATE_ROWS}, "
    f"Got {partial_dates.iloc[0]}" )


# Week number assertions

assert df18["week_number"].min() == EXPECTED_MIN_WEEK, (
    f"Minimum week number mismatch: EXPECTED {EXPECTED_MIN_WEEK}, "
    f"Got {df18['week_number'].min()}"
)

assert df18["week_number"].max() == EXPECTED_MAX_WEEK, (
    f"Maximum week number mismatch: EXPECTED {EXPECTED_MAX_WEEK}, "
    f"Got {df18['week_number'].max()}"
)

week_5_dates = df18.loc[
    df18["week_number"] == EXPECTED_MIN_WEEK, "date"].nunique()

assert week_5_dates == EXPECTED_WEEK_5_DATES, (
    f"Week 5 date count mismatch: EXPECTED {EXPECTED_WEEK_5_DATES}, "
    f"Got {week_5_dates}"
)

week_53_dates = df18.loc[
    df18["week_number"] == EXPECTED_MAX_WEEK, "date"
].nunique()

assert week_53_dates == EXPECTED_WEEK_53_DATES, (
    f"Week 53 date count mismatch: EXPECTED {EXPECTED_WEEK_53_DATES}, "
    f"Got {week_53_dates}"
)


# Missing value assertion

actual_missing_values = df18.isna().sum().sum()

assert actual_missing_values == EXPECTED_MISSING_VALUES, (
    f"Missing value count mismatch: EXPECTED {EXPECTED_MISSING_VALUES}, "
    f"Got {actual_missing_values}"
)


# Rental count assertion

assert df18["rented_bike_count"].min() >= EXPECTED_MIN_RENTALS, (
    f"Minimum rental count mismatch: EXPECTED at least {EXPECTED_MIN_RENTALS}, "
    f"Got {df18['rented_bike_count'].min()}"
)

# The open and closed records reconcile 
total_rows = len(df18) + len(closed18)

assert total_rows == EXPECTED_FULL_HOURS, ( 
    f"Total row count mismatch: EXPECTED {EXPECTED_FULL_HOURS}, " 
    f"Got {total_rows}") 

# Verify the number of dates across both datasets 
all_dates = set(df18["date"].unique()) | set(closed18["date"].unique()) 

assert len(all_dates) == EXPECTED_TOTAL_DATES, ( 
    f"Total unique date count mismatch: EXPECTED {EXPECTED_TOTAL_DATES}, " 
    f"Got {len(all_dates)}")

# Verify that closed dates minus dates in df18 equal the expected absent dates 
closed_dates = set(closed18["date"].dt.strftime("%Y-%m-%d")) 
open_dates = set(df18["date"].dt.strftime("%Y-%m-%d"))
actual_absent_dates = closed_dates - open_dates 

assert actual_absent_dates == EXPECTED_ABSENT_DATES, ( 
    f"Absent dates mismatch: EXPECTED {EXPECTED_ABSENT_DATES}, "
    f"Got {actual_absent_dates}" 
)

print("All 2018 data validation assertions passed.")

All 2018 data validation assertions passed.


## **Step 3: Data Audit**

In [4]:
# Calculate audit metrics

missing_values = df18.isna().sum().sum()
duplicate_rows = df18.duplicated().sum()

lowest_rentals = df18["rented_bike_count"].min()
highest_rentals = df18["rented_bike_count"].max()

duplicate_date_hour = df18.duplicated(subset=["date", "hour"]).sum()

closed_rows = len(closed18)
closed_dates_count = closed18["date"].nunique()
closed_hours_in_df18 = (
    df18["functioning_day"] == CLOSED_LABEL
).sum()

season_dates = (
    df18.groupby("seasons")["date"]
    .nunique()
    .reindex(["Winter", "Spring", "Summer", "Autumn"])
)

season_result = "; ".join(
    f"{season}: {count} dates"
    for season, count in season_dates.items()
)

# rows_per_date and partial_dates come from Step 2
if len(partial_dates) == 1:
    partial_date = partial_dates.index[0]
    partial_rows = partial_dates.iloc[0]

    hours_result = (
        f"{EXPECTED_DISTINCT_HOURS} rows on all dates except "
        f"{partial_date.strftime('%d %B')} ({partial_rows} rows)"
    )
else:
    hours_result = (
        f"{len(partial_dates)} dates have fewer than {EXPECTED_DISTINCT_HOURS} rows"
    )


# Assert no duplicate rows

assert duplicate_rows == EXPECTED_DUPLICATE_ROW, (
    f"Duplicate rows: EXPECTED {EXPECTED_DUPLICATE_ROW}, "
    f"Got {duplicate_rows}"
)

# Assert no duplicated date and hour pairs

assert duplicate_date_hour == EXPECTED_DUPLICATE_DATE_HOUR, (
    f"Duplicate date and hour pairs: EXPECTED {EXPECTED_DUPLICATE_DATE_HOUR}, "
    f"Got {duplicate_date_hour}"
)


# Build the audit rows. The holiday row is filled in Step 4.

audit_rows = {
    "Source file and date range": (
        f"{SOURCE_FILE}, {df18['date'].min():%d %B %Y} to {df18['date'].max():%d %B %Y}"
    ),
    "Records in scope": f"{len(df18):,} open rows across {df18['date'].nunique()} dates",
    "Hours per date": hours_result,
    "Missing values": f"{missing_values:,}",
    "Duplicate rows": f"{duplicate_rows:,}",
    "Duplicate date and hour pairs": f"{duplicate_date_hour:,}",
    "Closed hours": (
        f"{closed_rows:,} rows across {closed_dates_count} dates, "
        f"{closed_hours_in_df18} inside df18"
    ),
    "Rentals per hour": f"Lowest: {lowest_rentals:,}; highest: {highest_rentals:,}",
    "Dates by season": season_result,
    "Holiday flag": "Reconciled in Step 4",
}


def to_markdown_table(rows: dict) -> str:
    """Return a two column Markdown table from a dict of check and result."""
    lines = ["| Check | Result |", "|:---|:---|"]
    lines += [f"| {check} | {result} |" for check, result in rows.items()]
    return "\n".join(lines)


display(Markdown("### **2018 Data Audit**\n\n" + to_markdown_table(audit_rows)))

### **2018 Data Audit**

| Check | Result |
|:---|:---|
| Source file and date range | SeoulBikeData.csv, 01 January 2018 to 30 November 2018 |
| Records in scope | 7,721 open rows across 322 dates |
| Hours per date | 24 rows on all dates except 06 October (17 rows) |
| Missing values | 0 |
| Duplicate rows | 0 |
| Duplicate date and hour pairs | 0 |
| Closed hours | 295 rows across 13 dates, 0 inside df18 |
| Rentals per hour | Lowest: 2; highest: 3,556 |
| Dates by season | Winter: 59 dates; Spring: 90 dates; Summer: 92 dates; Autumn: 81 dates |
| Holiday flag | Reconciled in Step 4 |

## **Step 4: Holiday Reconciliation**

In [5]:
# Expected values

EXPECTED_RAW_HOLIDAY_DATES = 13
EXPECTED_VERIFIED_IN_DF18 = 16
EXPECTED_MISMATCH_COUNT = 5

# One reason for every date where the raw flag and the verified list disagree
EXPECTED_MISMATCH_REASONS = {
    "2018-05-01": "Raw false flag: Labour Day is not on the verified list",
    "2018-05-05": "Raw miss: Children's Day",
    "2018-05-07": "Raw miss: substitute holiday for Children's Day",
    "2018-06-13": "Raw miss: local election day",
    "2018-09-26": "Raw miss: substitute holiday for Chuseok",
}

# Verified dates with no row in df18
EXPECTED_VERIFIED_NOT_IN_DF18 = {
    "2017-12-25": "Outside 2018",
    "2018-10-09": "Centre closed, so the date sits in closed18",
}


# Convert the verified holiday dates to datetime

holiday_dates = pd.DatetimeIndex(sorted(HOLIDAYS))


# One row per date: raw flag against verified list

daily_holiday = (
    df18.assign(raw_holiday=df18["holiday"].eq("Holiday"))
    .groupby("date", as_index=False)
    .agg(raw_holiday=("raw_holiday", "max"))
)

daily_holiday["verified_official"] = daily_holiday["date"].isin(holiday_dates)
daily_holiday["match"] = (
    daily_holiday["raw_holiday"] == daily_holiday["verified_official"]
)


# Holiday table: every date flagged by either source

holiday_table = daily_holiday.loc[
    daily_holiday["raw_holiday"] | daily_holiday["verified_official"]
].copy()

holiday_table["reason"] = (
    holiday_table["date"].dt.strftime("%Y-%m-%d")
    .map(EXPECTED_MISMATCH_REASONS)
    .fillna("")
)

holiday_table = holiday_table.rename(
    columns={
        "raw_holiday": "raw flag",
        "verified_official": "verified official",
    }
)
display(holiday_table)

mismatches = holiday_table.loc[~holiday_table["match"]]

print("\nHoliday mismatches:")
print(mismatches.to_string(index=False))


# Assertions

raw_holiday_dates = int(daily_holiday["raw_holiday"].sum())
verified_in_df18 = int(daily_holiday["verified_official"].sum())
actual_mismatch_dates = set(mismatches["date"].dt.strftime("%Y-%m-%d"))

assert raw_holiday_dates == EXPECTED_RAW_HOLIDAY_DATES, (
    f"Raw holiday date count mismatch: EXPECTED {EXPECTED_RAW_HOLIDAY_DATES}, "
    f"Got {raw_holiday_dates}"
)

assert verified_in_df18 == EXPECTED_VERIFIED_IN_DF18, (
    f"Verified holidays inside df18 mismatch: EXPECTED {EXPECTED_VERIFIED_IN_DF18}, "
    f"Got {verified_in_df18}"
)

assert len(mismatches) == EXPECTED_MISMATCH_COUNT, (
    f"Mismatch count: EXPECTED {EXPECTED_MISMATCH_COUNT}, "
    f"Got {len(mismatches)}"
)

assert actual_mismatch_dates == set(EXPECTED_MISMATCH_REASONS), (
    f"Mismatch dates: EXPECTED {sorted(EXPECTED_MISMATCH_REASONS)}, "
    f"Got {sorted(actual_mismatch_dates)}"
)

# Verified dates with no row in df18 must be the two we can explain

df18_dates = set(df18["date"])
actual_verified_not_in_df18 = {
    day.strftime("%Y-%m-%d") for day in holiday_dates if day not in df18_dates
}

assert actual_verified_not_in_df18 == set(EXPECTED_VERIFIED_NOT_IN_DF18), (
    f"Verified dates missing from df18: EXPECTED {sorted(EXPECTED_VERIFIED_NOT_IN_DF18)}, "
    f"Got {sorted(actual_verified_not_in_df18)}"
)

# 9 October must be a fully closed date, not a gap
assert "2018-10-09" in EXPECTED_ABSENT_DATES, "9 October should be a closed date"

print("\nVerified dates with no rows in df18:")
for day, reason in EXPECTED_VERIFIED_NOT_IN_DF18.items():
    print(f"  {day}: {reason}")

print("\nAll holiday reconciliation assertions passed.")


# Fill the holiday row and show the final audit table

audit_rows["Holiday flag"] = (
    f"{raw_holiday_dates} raw flags, {verified_in_df18} verified dates inside df18, "
    f"{len(mismatches)} mismatches"
)

display(Markdown("### **2018 Data Audit (final)**\n\n" + to_markdown_table(audit_rows)))


# Save the tables

TABLE_DIR.mkdir(parents=True, exist_ok=True)

holiday_table.to_csv(TABLE_DIR / "y18_holiday_check.csv", index=False)
mismatches.to_csv(TABLE_DIR / "y18_holiday_mismatches.csv", index=False)
(TABLE_DIR / "y18_data_audit.md").write_text(
    to_markdown_table(audit_rows) + "\n", encoding="utf-8"
)

print("Tables saved to", TABLE_DIR.resolve())

,date,raw flag,verified official,match,reason
0,2018-01-01,True,True,True,
45,2018-02-15,True,True,True,
46,2018-02-16,True,True,True,
47,2018-02-17,True,True,True,
59,2018-03-01,True,True,True,
119,2018-05-01,True,False,False,Raw false flag: Labour Day is not on the verif...
123,2018-05-05,False,True,False,Raw miss: Children's Day
125,2018-05-07,False,True,False,Raw miss: substitute holiday for Children's Day
139,2018-05-22,True,True,True,
154,2018-06-06,True,True,True,



Holiday mismatches:
      date  raw flag  verified official  match                                                 reason
2018-05-01      True              False  False Raw false flag: Labour Day is not on the verified list
2018-05-05     False               True  False                               Raw miss: Children's Day
2018-05-07     False               True  False        Raw miss: substitute holiday for Children's Day
2018-06-13     False               True  False                           Raw miss: local election day
2018-09-26     False               True  False               Raw miss: substitute holiday for Chuseok

Verified dates with no rows in df18:
  2017-12-25: Outside 2018
  2018-10-09: Centre closed, so the date sits in closed18

All holiday reconciliation assertions passed.


### **2018 Data Audit (final)**

| Check | Result |
|:---|:---|
| Source file and date range | SeoulBikeData.csv, 01 January 2018 to 30 November 2018 |
| Records in scope | 7,721 open rows across 322 dates |
| Hours per date | 24 rows on all dates except 06 October (17 rows) |
| Missing values | 0 |
| Duplicate rows | 0 |
| Duplicate date and hour pairs | 0 |
| Closed hours | 295 rows across 13 dates, 0 inside df18 |
| Rentals per hour | Lowest: 2; highest: 3,556 |
| Dates by season | Winter: 59 dates; Spring: 90 dates; Summer: 92 dates; Autumn: 81 dates |
| Holiday flag | 13 raw flags, 16 verified dates inside df18, 5 mismatches |

Tables saved to C:\Users\USER\OneDrive\Desktop\SeoulBikeRentals\reports\tables
